# [Super AI Engineer Season 6] Hackathon Week 6
## 5 Domains Hackathon: Sleep Staging Classification

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: Sleep Staging Classification
- Notebook: Fast LightGBM sensor-feature pipeline
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Configuration  
3. Data Loading & Initial Inspection  
4. Helper Functions  
5. Feature Engineering  
6. Build Train Features  
7. Sequence Context Features  
8. Model Matrix Preparation  
9. Group Validation  
10. Final LightGBM Training  
11. Build Test Features  
12. Prediction & Submission Generation

# 1. Setup & Imports
### 1.1 Prepare dependencies for the fast tabular ensemble

Import signal-processing utilities and LightGBM first. If LightGBM is missing, the notebook tries a small install without restarting and can fall back to a sklearn model.

In [1]:
import importlib.util
import os
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path


def try_import_dependencies():
    try:
        import numpy as np
        import pandas as pd
        from scipy.stats import kurtosis, skew
        from sklearn.ensemble import HistGradientBoostingClassifier
        from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
        from sklearn.model_selection import GroupKFold, GroupShuffleSplit, StratifiedKFold
        from sklearn.preprocessing import LabelEncoder
        from sklearn.utils.class_weight import compute_sample_weight
        from tqdm.notebook import tqdm
        try:
            import lightgbm as lgb
            lgb_error = None
        except Exception as exc:
            lgb = None
            lgb_error = exc
        return {
            "np": np,
            "pd": pd,
            "kurtosis": kurtosis,
            "skew": skew,
            "HistGradientBoostingClassifier": HistGradientBoostingClassifier,
            "accuracy_score": accuracy_score,
            "classification_report": classification_report,
            "confusion_matrix": confusion_matrix,
            "f1_score": f1_score,
            "GroupKFold": GroupKFold,
            "GroupShuffleSplit": GroupShuffleSplit,
            "StratifiedKFold": StratifiedKFold,
            "LabelEncoder": LabelEncoder,
            "compute_sample_weight": compute_sample_weight,
            "tqdm": tqdm,
            "lgb": lgb,
            "lgb_error": lgb_error,
        }, None
    except Exception as exc:
        return None, exc


deps, import_error = try_import_dependencies()
if deps is None:
    raise RuntimeError(f"Core dependency import failed: {import_error!r}")

globals().update(deps)

INSTALL_LIGHTGBM = os.environ.get("SLEEP_INSTALL_LIGHTGBM", "1") == "1"
if lgb is None and INSTALL_LIGHTGBM:
    print("LightGBM import failed:", repr(lgb_error))
    print("Installing lightgbm once without restarting the kernel...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        import lightgbm as lgb
        print("LightGBM installed and imported.")
    except Exception as exc:
        print("LightGBM install/import still failed. Falling back to sklearn HistGradientBoosting:", repr(exc))
        lgb = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 2. Configuration
### 2.1 Set paths and run controls

Prefer the Kaggle competition dataset path, allow `SLEEP_DATA_DIR` as an override, and define feature cache, output files, fold count, estimator count, and smoothing switches in one place.

In [2]:
COMPETITION_NAME = "super-ai-engineer-ss-6-individual-sleep-stage-classification"
LEGACY_COMPETITION_NAME = "super-ai-engineer-ss-6-sleep-stage-classification"
WINDOW_SIZE = 480          # 30 seconds * 16 Hz
SAMPLE_RATE = 16
EPS = 1e-9

DATA_DIR_CANDIDATES = []
if os.environ.get("SLEEP_DATA_DIR"):
    DATA_DIR_CANDIDATES.append(Path(os.environ["SLEEP_DATA_DIR"]))

DATA_DIR_CANDIDATES.extend([
    Path(f"/kaggle/input/competitions/{COMPETITION_NAME}"),
    Path(f"/kaggle/input/{COMPETITION_NAME}"),
    Path(f"/kaggle/input/competitions/{LEGACY_COMPETITION_NAME}"),
    Path(f"/kaggle/input/{LEGACY_COMPETITION_NAME}"),
    Path.cwd() / COMPETITION_NAME,
    Path.cwd() / LEGACY_COMPETITION_NAME,
])

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
FEATURE_CACHE_DIR = WORK_DIR / "sleep_feature_cache_lv2"
SUBMISSION_PATH = WORK_DIR / "sleep_stage_lv2_submission.csv"
RAW_SUBMISSION_PATH = WORK_DIR / "sleep_stage_lv2_raw_submission.csv"

RUN_VALIDATION = os.environ.get("RUN_VALIDATION", "0") == "1"
VALIDATION_TIME_LIMIT = int(os.environ.get("SLEEP_VAL_TIME_LIMIT", "240"))
FINAL_TIME_LIMIT = int(os.environ.get("SLEEP_FINAL_TIME_LIMIT", "1500"))
LIGHTGBM_FOLDS = int(os.environ.get("SLEEP_LGBM_FOLDS", "5"))
LIGHTGBM_ESTIMATORS = int(os.environ.get("SLEEP_LGBM_ESTIMATORS", "650"))
LIGHTGBM_LEARNING_RATE = float(os.environ.get("SLEEP_LGBM_LEARNING_RATE", "0.035"))
SMOOTH_PROBABILITIES = os.environ.get("SMOOTH_PROBABILITIES", "1") == "1"
SMOOTH_ISOLATED_STAGES = os.environ.get("SMOOTH_ISOLATED_STAGES", "1") == "1"
USE_FEATURE_CACHE = os.environ.get("USE_FEATURE_CACHE", "1") == "1"


def looks_like_dataset(path: Path) -> bool:
    return path.exists() and (path / "sample_submission.csv").exists() and (path / "train").exists()


def resolve_data_dir(candidates):
    for path in candidates:
        if looks_like_dataset(path):
            return path
    print("Checked dataset candidates:")
    for path in candidates:
        print(" -", path)
    raise FileNotFoundError("Dataset directory not found. Set SLEEP_DATA_DIR or attach the Kaggle competition dataset.")


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)
print("Model backend:", "LightGBM" if lgb is not None else "sklearn HistGradientBoosting fallback")
print("LightGBM folds:", LIGHTGBM_FOLDS)
print("LightGBM estimators:", LIGHTGBM_ESTIMATORS)

DATA_DIR: /kaggle/input/competitions/super-ai-engineer-ss-6-individual-sleep-stage-classification
WORK_DIR: /kaggle/working
Model backend: LightGBM
LightGBM folds: 5
LightGBM estimators: 650


# 3. Data Loading & Initial Inspection
### 3.1 Load train files and sample submission

Collect all training files, inspect sensor columns and subject counts, and check the test id format before feature extraction.

In [3]:
def collect_train_files(data_dir: Path):
    patterns = [
        data_dir / "train" / "train" / "*.csv",
        data_dir / "train" / "*.csv",
    ]
    files = []
    for pattern in patterns:
        files.extend(sorted(pattern.parent.glob(pattern.name)))
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError("No train csv files found under train/train or train.")
    return files

train_files = collect_train_files(DATA_DIR)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train files:", len(train_files))
print("Sample submission shape:", sample_submission.shape)
display(sample_submission.head())

preview = pd.read_csv(train_files[0], nrows=5)
print("Example train file:", train_files[0].name)
display(preview)

Train files: 83
Sample submission shape: (7832, 2)


,id,labels
0,test001_00000,W
1,test001_00001,W
2,test001_00002,W
3,test001_00003,NaN
4,test001_00004,NaN


Example train file: train001.csv


,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage
0,25.325870,-21.809247,-60.302750,4.940839,31.722653,0.064595,72.015570,1.050338,W
1,20.021505,-19.437787,-60.565345,7.408788,31.722647,0.064523,72.015802,1.050338,W
2,16.314478,-21.624667,-61.142561,4.105717,31.722735,0.064659,72.017417,1.050338,W
3,9.324392,-21.761314,-61.985822,3.972967,31.722564,0.064440,72.013801,1.050338,W
4,-1.014338,-19.055301,-59.934137,9.097628,31.722790,0.065397,72.018920,1.050338,W


# 4. Helper Functions
### 4.1 Standardize columns, labels, and paths

Provide helper functions for column aliases, label parsing, subject id parsing, and test segment path resolution.

In [4]:
COLUMN_ALIAS = {
    "bvp": "BVP",
    "ibi": "IBI",
    "eda": "EDA",
    "temp": "TEMP",
    "temperature": "TEMP",
    "accx": "ACC_X",
    "accy": "ACC_Y",
    "accz": "ACC_Z",
    "hr": "HR",
    "heartrate": "HR",
    "sleepstage": "Sleep_Stage",
    "stage": "Sleep_Stage",
    "label": "Sleep_Stage",
    "labels": "Sleep_Stage",
}

SIGNAL_COLUMNS = ["BVP", "IBI", "EDA", "TEMP", "ACC_X", "ACC_Y", "ACC_Z", "HR"]
FFT_COLUMNS = ["BVP", "EDA", "TEMP", "ACC_X", "ACC_Y", "ACC_Z", "HR"]
FFT_BANDS = {
    "very_low": (0.03, 0.15),
    "low": (0.15, 0.40),
    "mid": (0.40, 1.00),
    "high": (1.00, 4.00),
    "very_high": (4.00, 8.00),
}


def normalize_column_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for col in df.columns:
        key = normalize_column_name(col)
        if key in COLUMN_ALIAS:
            rename[col] = COLUMN_ALIAS[key]
    return df.rename(columns=rename)


def parse_label(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        if len(value) == 0:
            return np.nan
        value = value[0]
    if pd.isna(value):
        return np.nan
    text = str(value).strip().strip("'").strip('"')
    if text.lower() in {"", "nan", "none", "null"}:
        return np.nan
    if text.startswith("[") and text.endswith("]"):
        text = text[1:-1].split(",")[0].strip().strip("'").strip('"')
    upper = text.upper()
    if upper in {"W", "R", "N1", "N2", "N3"}:
        return upper
    try:
        number = float(text)
        if number.is_integer():
            return int(number)
        return number
    except ValueError:
        return text


def parse_subject_id_from_path(path) -> str:
    stem = Path(path).stem
    return re.split(r"[_\-]", stem)[0]


def split_test_id(id_value):
    text = str(id_value)
    parts = text.rsplit("_", 1)
    if len(parts) == 2 and re.fullmatch(r"\d+", parts[1]):
        return parts[0], int(parts[1])
    match = re.search(r"(.+?)[_\-](\d+)$", text)
    if match:
        return match.group(1), int(match.group(2))
    return text, np.nan


def resolve_test_segment_path(id_value, data_dir: Path) -> Path:
    subject_id, segment_idx = split_test_id(id_value)
    id_filename = f"{id_value}.csv"
    candidate_filenames = [id_filename]
    if not pd.isna(segment_idx):
        candidate_filenames.append(f"{subject_id}_{int(segment_idx)}.csv")

    roots = [
        data_dir / "test_segment" / "test_segment",
        data_dir / "test_segment",
    ]
    candidates = []
    for root in roots:
        for filename in candidate_filenames:
            candidates.extend([
                root / str(subject_id) / filename,
                root / filename,
            ])
    for path in candidates:
        if path.exists():
            return path

    for filename in candidate_filenames:
        matches = list(data_dir.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Cannot find test segment for id={id_value}")

# 5. Feature Engineering
### 5.1 Extract one feature row per 30-second segment

Create statistical, quantile, RMS, energy, slope, FFT, accelerometer magnitude, and sensor-correlation features for each complete segment.

In [5]:
def numeric_array(df: pd.DataFrame, column: str):
    if column not in df.columns:
        arr = np.full(len(df), np.nan, dtype=np.float32)
    else:
        arr = pd.to_numeric(df[column], errors="coerce").to_numpy(dtype=np.float32)
    finite_mask = np.isfinite(arr)
    missing_frac = 1.0 - float(finite_mask.mean()) if len(arr) else 1.0
    if finite_mask.any():
        fill_value = float(np.nanmedian(arr))
    else:
        fill_value = 0.0
    filled = np.where(finite_mask, arr, fill_value).astype(np.float32)
    return arr, filled, missing_frac


def safe_corr(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if len(a) < 3 or np.std(a) < EPS or np.std(b) < EPS:
        return 0.0
    value = np.corrcoef(a, b)[0, 1]
    if not np.isfinite(value):
        return 0.0
    return float(value)


def add_stats_features(features: dict, arr: np.ndarray, prefix: str, missing_frac: float = 0.0):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.size == 0:
        arr = np.zeros(WINDOW_SIZE, dtype=np.float32)
    mean = float(np.mean(arr))
    std = float(np.std(arr))
    diff = np.diff(arr) if arr.size > 1 else np.array([0.0], dtype=np.float32)
    centered = arr - mean
    x = np.arange(arr.size, dtype=np.float32)
    slope = float(np.dot(x - x.mean(), centered) / (np.dot(x - x.mean(), x - x.mean()) + EPS))
    q10, q25, q50, q75, q90 = np.quantile(arr, [0.10, 0.25, 0.50, 0.75, 0.90])

    features[f"{prefix}_missing_frac"] = missing_frac
    features[f"{prefix}_mean"] = mean
    features[f"{prefix}_std"] = std
    features[f"{prefix}_var"] = float(np.var(arr))
    features[f"{prefix}_min"] = float(np.min(arr))
    features[f"{prefix}_max"] = float(np.max(arr))
    features[f"{prefix}_median"] = float(q50)
    features[f"{prefix}_q10"] = float(q10)
    features[f"{prefix}_q25"] = float(q25)
    features[f"{prefix}_q75"] = float(q75)
    features[f"{prefix}_q90"] = float(q90)
    features[f"{prefix}_iqr"] = float(q75 - q25)
    features[f"{prefix}_range"] = float(np.max(arr) - np.min(arr))
    features[f"{prefix}_mad"] = float(np.mean(np.abs(arr - mean)))
    features[f"{prefix}_rms"] = float(np.sqrt(np.mean(arr ** 2)))
    features[f"{prefix}_energy"] = float(np.mean(arr ** 2))
    features[f"{prefix}_first"] = float(arr[0])
    features[f"{prefix}_last"] = float(arr[-1])
    features[f"{prefix}_delta"] = float(arr[-1] - arr[0])
    features[f"{prefix}_slope"] = slope
    features[f"{prefix}_absdiff_mean"] = float(np.mean(np.abs(diff)))
    features[f"{prefix}_absdiff_std"] = float(np.std(np.abs(diff)))
    features[f"{prefix}_absdiff_max"] = float(np.max(np.abs(diff)))
    features[f"{prefix}_zcr"] = float(np.mean(np.diff(np.signbit(centered)) != 0)) if arr.size > 1 else 0.0
    if std > EPS:
        features[f"{prefix}_skew"] = float(skew(arr, bias=False))
        features[f"{prefix}_kurtosis"] = float(kurtosis(arr, bias=False))
    else:
        features[f"{prefix}_skew"] = 0.0
        features[f"{prefix}_kurtosis"] = 0.0


def add_fft_features(features: dict, arr: np.ndarray, prefix: str):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.size < 4 or np.std(arr) < EPS:
        for band in FFT_BANDS:
            features[f"{prefix}_fft_{band}_power"] = 0.0
            features[f"{prefix}_fft_{band}_rel"] = 0.0
        features[f"{prefix}_fft_dom_freq"] = 0.0
        features[f"{prefix}_fft_centroid"] = 0.0
        features[f"{prefix}_fft_entropy"] = 0.0
        return

    window = np.hanning(arr.size).astype(np.float32)
    spectrum = np.fft.rfft((arr - np.mean(arr)) * window)
    power = np.abs(spectrum) ** 2
    freqs = np.fft.rfftfreq(arr.size, d=1.0 / SAMPLE_RATE)
    power[0] = 0.0
    total_power = float(power.sum() + EPS)

    for band_name, (low, high) in FFT_BANDS.items():
        mask = (freqs >= low) & (freqs < high)
        band_power = float(power[mask].sum()) if mask.any() else 0.0
        features[f"{prefix}_fft_{band_name}_power"] = band_power
        features[f"{prefix}_fft_{band_name}_rel"] = band_power / total_power

    dominant_idx = int(np.argmax(power))
    features[f"{prefix}_fft_dom_freq"] = float(freqs[dominant_idx])
    features[f"{prefix}_fft_centroid"] = float((freqs * power).sum() / total_power)
    prob = power / total_power
    features[f"{prefix}_fft_entropy"] = float(-(prob * np.log(prob + EPS)).sum())


def extract_segment_features(segment_df: pd.DataFrame, subject_id: str, segment_idx: int, n_segments: int):
    segment_df = standardize_columns(segment_df)
    features = {
        "subject_id": str(subject_id),
        "segment_idx": int(segment_idx),
        "n_segments": int(n_segments),
        "segment_pos": float(segment_idx / max(n_segments - 1, 1)),
    }

    arrays = {}
    for column in SIGNAL_COLUMNS:
        _, filled, missing_frac = numeric_array(segment_df, column)
        arrays[column] = filled
        add_stats_features(features, filled, column.lower(), missing_frac)
        if column in FFT_COLUMNS:
            add_fft_features(features, filled, column.lower())

    if all(column in arrays for column in ["ACC_X", "ACC_Y", "ACC_Z"]):
        acc_mag = np.sqrt(arrays["ACC_X"] ** 2 + arrays["ACC_Y"] ** 2 + arrays["ACC_Z"] ** 2)
        arrays["ACC_MAG"] = acc_mag
        add_stats_features(features, acc_mag, "acc_mag", 0.0)
        add_fft_features(features, acc_mag, "acc_mag")
        features["acc_xy_corr"] = safe_corr(arrays["ACC_X"], arrays["ACC_Y"])
        features["acc_xz_corr"] = safe_corr(arrays["ACC_X"], arrays["ACC_Z"])
        features["acc_yz_corr"] = safe_corr(arrays["ACC_Y"], arrays["ACC_Z"])

    features["bvp_hr_corr"] = safe_corr(arrays.get("BVP", []), arrays.get("HR", []))
    features["eda_temp_corr"] = safe_corr(arrays.get("EDA", []), arrays.get("TEMP", []))
    features["hr_temp_corr"] = safe_corr(arrays.get("HR", []), arrays.get("TEMP", []))
    features["bvp_hr_mean_ratio"] = float(features.get("bvp_mean", 0.0) / (features.get("hr_mean", 0.0) + EPS))
    features["eda_temp_mean_ratio"] = float(features.get("eda_mean", 0.0) / (features.get("temp_mean", 0.0) + EPS))
    return features

# 6. Build Train Features
### 6.1 Convert train signals into a feature table

Read each subject file, split it into 480-row windows, attach labels, and cache the resulting feature table under the working directory.

In [6]:
def build_train_features(train_files, use_cache=True):
    cache_path = FEATURE_CACHE_DIR / "train_features.pkl"
    if use_cache and cache_path.exists():
        print("Load cached train features:", cache_path)
        return pd.read_pickle(cache_path)

    rows = []
    failed_files = []
    for path in tqdm(train_files, desc="Extract train features"):
        try:
            df = standardize_columns(pd.read_csv(path))
            if "Sleep_Stage" not in df.columns:
                raise ValueError("Sleep_Stage column not found")
            n_segments = len(df) // WINDOW_SIZE
            subject_id = parse_subject_id_from_path(path)
            if n_segments == 0:
                raise ValueError("file is shorter than one full segment")

            for segment_idx in range(n_segments):
                start = segment_idx * WINDOW_SIZE
                segment = df.iloc[start:start + WINDOW_SIZE].copy()
                label = parse_label(segment["Sleep_Stage"].iloc[0])
                if pd.isna(label):
                    continue
                features = extract_segment_features(segment.drop(columns=["Sleep_Stage"], errors="ignore"), subject_id, segment_idx, n_segments)
                features["label"] = label
                features["source_file"] = Path(path).name
                rows.append(features)
        except Exception as exc:
            failed_files.append((str(path), repr(exc)))

    train_features = pd.DataFrame(rows)
    if train_features.empty:
        raise RuntimeError("No train features were created.")

    train_features = train_features.replace([np.inf, -np.inf], np.nan)
    train_features = train_features.dropna(subset=["label"]).reset_index(drop=True)
    train_features.to_pickle(cache_path)

    print("Train feature shape:", train_features.shape)
    if failed_files:
        print("Failed files:", len(failed_files))
        display(pd.DataFrame(failed_files, columns=["path", "error"]).head(20))
    return train_features

train_features = build_train_features(train_files, use_cache=USE_FEATURE_CACHE)
display(train_features.head())
display(train_features["label"].value_counts(dropna=False).sort_index())

Extract train features:   0%|          | 0/83 [00:00<?, ?it/s]

Train feature shape: (66745, 352)


,subject_id,segment_idx,n_segments,segment_pos,bvp_missing_frac,bvp_mean,bvp_std,bvp_var,bvp_min,bvp_max,bvp_median,bvp_q10,bvp_q25,bvp_q75,bvp_q90,bvp_iqr,bvp_range,bvp_mad,bvp_rms,bvp_energy,bvp_first,bvp_last,bvp_delta,bvp_slope,bvp_absdiff_mean,bvp_absdiff_std,bvp_absdiff_max,bvp_zcr,bvp_skew,bvp_kurtosis,bvp_fft_very_low_power,bvp_fft_very_low_rel,bvp_fft_low_power,bvp_fft_low_rel,bvp_fft_mid_power,bvp_fft_mid_rel,bvp_fft_high_power,bvp_fft_high_rel,bvp_fft_very_high_power,bvp_fft_very_high_rel,bvp_fft_dom_freq,bvp_fft_centroid,bvp_fft_entropy,ibi_missing_frac,ibi_mean,ibi_std,ibi_var,ibi_min,ibi_max,ibi_median,ibi_q10,ibi_q25,ibi_q75,ibi_q90,ibi_iqr,ibi_range,ibi_mad,ibi_rms,ibi_energy,ibi_first,...,hr_fft_low_power,hr_fft_low_rel,hr_fft_mid_power,hr_fft_mid_rel,hr_fft_high_power,hr_fft_high_rel,hr_fft_very_high_power,hr_fft_very_high_rel,hr_fft_dom_freq,hr_fft_centroid,hr_fft_entropy,acc_mag_missing_frac,acc_mag_mean,acc_mag_std,acc_mag_var,acc_mag_min,acc_mag_max,acc_mag_median,acc_mag_q10,acc_mag_q25,acc_mag_q75,acc_mag_q90,acc_mag_iqr,acc_mag_range,acc_mag_mad,acc_mag_rms,acc_mag_energy,acc_mag_first,acc_mag_last,acc_mag_delta,acc_mag_slope,acc_mag_absdiff_mean,acc_mag_absdiff_std,acc_mag_absdiff_max,acc_mag_zcr,acc_mag_skew,acc_mag_kurtosis,acc_mag_fft_very_low_power,acc_mag_fft_very_low_rel,acc_mag_fft_low_power,acc_mag_fft_low_rel,acc_mag_fft_mid_power,acc_mag_fft_mid_rel,acc_mag_fft_high_power,acc_mag_fft_high_rel,acc_mag_fft_very_high_power,acc_mag_fft_very_high_rel,acc_mag_fft_dom_freq,acc_mag_fft_centroid,acc_mag_fft_entropy,acc_xy_corr,acc_xz_corr,acc_yz_corr,bvp_hr_corr,eda_temp_corr,hr_temp_corr,bvp_hr_mean_ratio,eda_temp_mean_ratio,label,source_file
0,train001,0,743,0.000000,0.0,2.816168,155.888290,24301.160156,-1212.411743,665.980835,1.940699,-69.096877,-11.156508,34.338478,132.155838,45.494987,1878.392578,75.504501,155.913742,24309.093750,25.325871,107.186371,81.860504,0.037286,39.512680,85.481689,942.055298,0.154489,-2.023790,14.853420,3.490774e+06,0.002550,39467048.0,0.028826,472896736.0,0.345398,810292608.0,0.591828,42987180.0,0.031397,1.066667,1.426539,4.347534,0.0,0.962701,4.305011e-02,1.853312e-03,0.916946,1.055185,0.942215,0.942215,0.942215,0.942215,1.050338,1.192093e-07,0.138239,3.395651e-02,0.963663,0.928646,1.050338,...,13.685002,0.000342,20.070381,0.000502,144.880402,0.003625,14.654895,0.000367,0.033333,0.042306,0.244196,0.0,64.454544,0.963677,0.928674,57.818638,74.209122,64.458164,63.884904,64.191477,64.735386,64.986306,0.543909,16.390484,0.460253,64.461754,4155.317383,64.315453,64.761887,0.446434,-0.000421,0.543289,1.046693,14.385139,0.432150,2.108316,41.626549,266.389587,0.003095,242.420761,0.002817,2623.713379,0.030487,53667.371094,0.623609,29258.837891,0.339985,1.800000,3.218253,4.928240,-0.671302,-0.733889,0.707053,0.035622,-0.158873,-0.498294,0.037028,0.002028,W,train001.csv
1,train001,1,743,0.001348,0.0,-2.661875,123.138374,15163.058594,-729.854614,466.121246,1.047080,-81.579884,-28.886622,38.463121,101.541608,67.349744,1195.975830,68.947769,123.167130,15170.142578,107.752388,50.166641,-57.585747,0.037157,32.984043,49.896038,552.241699,0.162839,-1.677215,9.824995,2.171043e+06,0.002803,70674088.0,0.091262,365688768.0,0.472215,327510272.0,0.422915,8367994.0,0.010806,0.800000,1.122219,4.155989,0.0,0.942215,1.192093e-07,1.421085e-14,0.942215,0.942215,0.942215,0.942215,0.942215,0.942215,0.942215,0.000000e+00,0.000000,1.192093e-07,0.942215,0.887769,0.942215,...,22.112608,0.000656,31.503883,0.000934,104.575256,0.003100,11.109816,0.000329,0.033333,0.044113,0.406022,0.0,64.507668,0.311741,0.097182,63.106846,65.868874,64.500011,64.146620,64.311747,64.699518,64.881956,0.387772,2.762028,0.237214,64.508415,4161.335449,64.392632,64.346413,-0.046219,-0.000064,0.306524,0.274060,2.060898,0.450939,0.140169,1.619186,87.294922,0.021115,199.340347,0.048217,292.460327,0.070741,1657.538086,0.400929,1897.614258,0.458999,5.300000,3.580858,4.886000,-0.755296,-0.521155,0.705173,0.045890,0.080883,0.214350,-0.0322

label
N1     7753
N2    33786
N3     2345
R      7033
W     15828
Name: count, dtype: int64

# 7. Sequence Context Features
### 7.1 Add subject-level context

Add subject-relative features, lag and lead features, and rolling context so the model can use sleep-stage continuity without reading test labels.

In [7]:
PREFERRED_CONTEXT_COLUMNS = [
    "bvp_mean", "bvp_std", "bvp_rms", "bvp_absdiff_mean", "bvp_fft_dom_freq",
    "ibi_mean", "ibi_std", "ibi_absdiff_mean",
    "hr_mean", "hr_std", "hr_slope",
    "eda_mean", "eda_std", "eda_slope",
    "temp_mean", "temp_std", "temp_slope",
    "acc_mag_mean", "acc_mag_std", "acc_mag_rms", "acc_mag_absdiff_mean", "acc_mag_fft_dom_freq",
    "acc_x_std", "acc_y_std", "acc_z_std",
]
CONTEXT_SHIFTS = [1, 2, 3]
CONTEXT_ROLL_WINDOWS = [3, 5, 10, 60]


def add_sequence_context(df: pd.DataFrame, context_columns=None) -> pd.DataFrame:
    df = df.copy()
    if "subject_id" not in df.columns or "segment_idx" not in df.columns:
        return df

    df["_original_order"] = np.arange(len(df))
    df = df.sort_values(["subject_id", "segment_idx", "_original_order"]).reset_index(drop=True)

    group_idx = df.groupby("subject_id", sort=False)["segment_idx"]
    min_idx = group_idx.transform("min")
    max_idx = group_idx.transform("max")
    span = (max_idx - min_idx).replace(0, 1)
    df["n_segments"] = group_idx.transform("count").astype(int)
    df["segment_pos"] = ((df["segment_idx"] - min_idx) / span).astype(np.float32)

    if context_columns is None:
        context_columns = [col for col in PREFERRED_CONTEXT_COLUMNS if col in df.columns]
    context_columns = [col for col in context_columns if col in df.columns]

    context_frames = []
    grouped = df.groupby("subject_id", sort=False)

    for col in context_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float32)
        group = grouped[col]

        subject_mean = group.transform("mean")
        subject_std = group.transform("std").replace(0, np.nan)
        feature_block = {
            f"{col}_subject_z": ((df[col] - subject_mean) / subject_std).astype(np.float32),
            f"{col}_subject_delta_mean": (df[col] - subject_mean).astype(np.float32),
        }

        for shift in CONTEXT_SHIFTS:
            lag = group.shift(shift)
            lead = group.shift(-shift)
            feature_block[f"{col}_lag{shift}"] = lag.astype(np.float32)
            feature_block[f"{col}_lead{shift}"] = lead.astype(np.float32)
            feature_block[f"{col}_diff_lag{shift}"] = (df[col] - lag).astype(np.float32)
            feature_block[f"{col}_diff_lead{shift}"] = (lead - df[col]).astype(np.float32)

        for window in CONTEXT_ROLL_WINDOWS:
            roll_mean = group.transform(lambda s, w=window: s.rolling(w, min_periods=1, center=True).mean())
            roll_std = group.transform(lambda s, w=window: s.rolling(w, min_periods=2, center=True).std())
            feature_block[f"{col}_roll{window}_mean"] = roll_mean.astype(np.float32)
            feature_block[f"{col}_roll{window}_std"] = roll_std.astype(np.float32)
            feature_block[f"{col}_roll{window}_delta"] = (df[col] - roll_mean).astype(np.float32)

        context_frames.append(pd.DataFrame(feature_block, index=df.index))

    if context_frames:
        df = pd.concat([df] + context_frames, axis=1).copy()

    df = df.sort_values("_original_order").drop(columns=["_original_order"]).reset_index(drop=True)
    return df


train_features = add_sequence_context(train_features)
print("Feature shape after sequence context:", train_features.shape)

Feature shape after sequence context: (66745, 1002)


# 8. Model Matrix Preparation
### 8.1 Build the numeric matrix for modeling

Drop metadata columns, keep numeric features, and fit a median imputer on train data so train and test share the same feature schema.

In [8]:
META_COLUMNS = {"label", "source_file", "subject_id", "id", "segment_path"}


def get_feature_columns(df: pd.DataFrame):
    feature_columns = [col for col in df.columns if col not in META_COLUMNS]
    numeric_feature_columns = []
    for col in feature_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_feature_columns.append(col)
    return numeric_feature_columns


def fit_imputer(df: pd.DataFrame, feature_columns):
    numeric = df[feature_columns].replace([np.inf, -np.inf], np.nan)
    medians = numeric.median(numeric_only=True)
    return medians


def apply_feature_imputer(df: pd.DataFrame, feature_columns, medians):
    out = df.copy()
    for col in feature_columns:
        if col not in out.columns:
            out[col] = np.nan
    out[feature_columns] = out[feature_columns].replace([np.inf, -np.inf], np.nan)
    out[feature_columns] = out[feature_columns].fillna(medians).fillna(0.0)
    return out

feature_columns = get_feature_columns(train_features)
feature_medians = fit_imputer(train_features, feature_columns)
train_model = apply_feature_imputer(train_features, feature_columns, feature_medians)[feature_columns + ["label"]]
groups = train_features["subject_id"].astype(str).reset_index(drop=True)

print("Rows:", len(train_model))
print("Features:", len(feature_columns))
display(train_model.head())

Rows: 66745
Features: 999


,segment_idx,n_segments,segment_pos,bvp_missing_frac,bvp_mean,bvp_std,bvp_var,bvp_min,bvp_max,bvp_median,bvp_q10,bvp_q25,bvp_q75,bvp_q90,bvp_iqr,bvp_range,bvp_mad,bvp_rms,bvp_energy,bvp_first,bvp_last,bvp_delta,bvp_slope,bvp_absdiff_mean,bvp_absdiff_std,bvp_absdiff_max,bvp_zcr,bvp_skew,bvp_kurtosis,bvp_fft_very_low_power,bvp_fft_very_low_rel,bvp_fft_low_power,bvp_fft_low_rel,bvp_fft_mid_power,bvp_fft_mid_rel,bvp_fft_high_power,bvp_fft_high_rel,bvp_fft_very_high_power,bvp_fft_very_high_rel,bvp_fft_dom_freq,bvp_fft_centroid,bvp_fft_entropy,ibi_missing_frac,ibi_mean,ibi_std,ibi_var,ibi_min,ibi_max,ibi_median,ibi_q10,ibi_q25,ibi_q75,ibi_q90,ibi_iqr,ibi_range,ibi_mad,ibi_rms,ibi_energy,ibi_first,ibi_last,...,acc_x_std_roll5_delta,acc_x_std_roll10_mean,acc_x_std_roll10_std,acc_x_std_roll10_delta,acc_x_std_roll60_mean,acc_x_std_roll60_std,acc_x_std_roll60_delta,acc_y_std_subject_z,acc_y_std_subject_delta_mean,acc_y_std_lag1,acc_y_std_lead1,acc_y_std_diff_lag1,acc_y_std_diff_lead1,acc_y_std_lag2,acc_y_std_lead2,acc_y_std_diff_lag2,acc_y_std_diff_lead2,acc_y_std_lag3,acc_y_std_lead3,acc_y_std_diff_lag3,acc_y_std_diff_lead3,acc_y_std_roll3_mean,acc_y_std_roll3_std,acc_y_std_roll3_delta,acc_y_std_roll5_mean,acc_y_std_roll5_std,acc_y_std_roll5_delta,acc_y_std_roll10_mean,acc_y_std_roll10_std,acc_y_std_roll10_delta,acc_y_std_roll60_mean,acc_y_std_roll60_std,acc_y_std_roll60_delta,acc_z_std_subject_z,acc_z_std_subject_delta_mean,acc_z_std_lag1,acc_z_std_lead1,acc_z_std_diff_lag1,acc_z_std_diff_lead1,acc_z_std_lag2,acc_z_std_lead2,acc_z_std_diff_lag2,acc_z_std_diff_lead2,acc_z_std_lag3,acc_z_std_lead3,acc_z_std_diff_lag3,acc_z_std_diff_lead3,acc_z_std_roll3_mean,acc_z_std_roll3_std,acc_z_std_roll3_delta,acc_z_std_roll5_mean,acc_z_std_roll5_std,acc_z_std_roll5_delta,acc_z_std_roll10_mean,acc_z_std_roll10_std,acc_z_std_roll10_delta,acc_z_std_roll60_mean,acc_z_std_roll60_std,acc_z_std_roll60_delta,label
0,0,743,0.000000,0.0,2.816168,155.888290,24301.160156,-1212.411743,665.980835,1.940699,-69.096877,-11.156508,34.338478,132.155838,45.494987,1878.392578,75.504501,155.913742,24309.093750,25.325871,107.186371,81.860504,0.037286,39.512680,85.481689,942.055298,0.154489,-2.023790,14.853420,3.490774e+06,0.002550,39467048.0,0.028826,472896736.0,0.345398,810292608.0,0.591828,42987180.0,0.031397,1.066667,1.426539,4.347534,0.0,0.962701,4.305011e-02,1.853312e-03,0.916946,1.055185,0.942215,0.942215,0.942215,0.942215,1.050338,1.192093e-07,0.138239,3.395651e-02,0.963663,0.928646,1.050338,0.942215,...,1.934227,2.029565,1.452289,2.017025,1.620483,0.732551,2.426108,-0.207101,-0.826927,0.228291,0.499949,0.000000,-0.807778,0.228291,0.192642,0.000000,-1.115084,0.228256,0.262302,0.000000,-1.045424,0.903837,0.571185,0.403889,0.666772,0.575956,0.640954,0.614159,0.456400,0.693567,0.525635,0.256084,0.782091,0.597413,2.004444,0.277797,2.474224,-1.564622e-07,-1.702457,0.277789,0.983103,-2.030283e-07,-3.193578,0.277682,1.536152,-0.000031,-2.640529,3.325452,1.203819,0.851229,2.544669,1.597954,1.632012,2.427844,1.248588,1.748837,2.031431,0.880020,2.145250,W
1,1,743,0.001348,0.0,-2.661875,123.138374,15163.058594,-729.854614,466.121246,1.047080,-81.579884,-28.886622,38.463121,101.541608,67.349744,1195.975830,68.947769,123.167130,15170.142578,107.752388,50.166641,-57.585747,0.037157,32.984043,49.896038,552.241699,0.162839,-1.677215,9.824995,2.171043e+06,0.002803,70674088.0,0.091262,365688768.0,0.472215,327510272.0,0.422915,8367994.0,0.010806,0.800000,1.122219,4.155989,0.0,0.942215,1.192093e-07,1.421085e-14,0.942215,0.942215,0.942215,0.942215,0.942215,0.942215,0.942215,0.000000e+00,0.000000,1.192093e-07,0.942215,0.887769,0.942215,0.942215,...,-0.179047,1.994556,1.301795,-0.383352,1.622734,0.720347,-0.011530,-0.409406,-1.634704,1.307726,0.192642,-0.807778,-0.307306,0.228291,0.262302,0.000000,-0.237646,0.228256,0.808178,0.000000,0.308229,0.666772,0.575956,-0.166824,0.565655,0.511908,-0.065706,0.608918,0.408418,-0.108970,0.526294,0.251806,-0.026345,0.090005,0.301987,4.176681,0.9831

# 9. Group Validation
### 9.1 Validate by subject when needed

Use subject id as the validation group to reduce leakage between nearby segments from the same person. This step is optional for faster final runs.

In [9]:
def make_fast_sleep_model(random_state: int, n_classes: int):
    if lgb is not None:
        return lgb.LGBMClassifier(
            objective="multiclass",
            num_class=n_classes,
            n_estimators=LIGHTGBM_ESTIMATORS,
            learning_rate=LIGHTGBM_LEARNING_RATE,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
            force_col_wise=True,
        )
    return HistGradientBoostingClassifier(
        max_iter=max(20, min(LIGHTGBM_ESTIMATORS, 250)),
        learning_rate=LIGHTGBM_LEARNING_RATE,
        l2_regularization=0.05,
        random_state=random_state,
    )


def fit_model(model, X, y, sample_weight=None):
    try:
        return model.fit(X, y, sample_weight=sample_weight)
    except TypeError:
        return model.fit(X, y)


def aligned_predict_proba(model, X, n_classes: int) -> np.ndarray:
    proba = model.predict_proba(X)
    out = np.zeros((len(X), n_classes), dtype=np.float32)
    model_classes = getattr(model, "classes_", np.arange(proba.shape[1]))
    for src_idx, class_idx in enumerate(model_classes):
        out[:, int(class_idx)] = proba[:, src_idx]
    return out


def run_group_validation(data: pd.DataFrame, groups: pd.Series):
    X = data.drop(columns=["label"]).astype(np.float32)
    encoder = LabelEncoder()
    y = encoder.fit_transform(data["label"].astype(str))
    n_classes = len(encoder.classes_)

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, valid_idx = next(splitter.split(X, y, groups=groups))
    sample_weight = compute_sample_weight(class_weight="balanced", y=y)

    model = make_fast_sleep_model(SEED + 1000, n_classes)
    fit_model(model, X.iloc[train_idx], y[train_idx], sample_weight=sample_weight[train_idx])
    valid_proba = aligned_predict_proba(model, X.iloc[valid_idx], n_classes)
    valid_pred = valid_proba.argmax(axis=1)
    valid_accuracy = accuracy_score(y[valid_idx], valid_pred)
    valid_f1 = f1_score(y[valid_idx], valid_pred, average="weighted")
    print(f"Validation accuracy: {valid_accuracy:.5f}")
    print(f"Validation weighted F1: {valid_f1:.5f}")
    return model, valid_f1, encoder.inverse_transform(valid_pred)


if RUN_VALIDATION:
    predictor_val, valid_score, valid_pred = run_group_validation(train_model, groups)
else:
    predictor_val, valid_score, valid_pred = None, None, None
    print("Validation skipped. Set RUN_VALIDATION=1 to run a grouped holdout check.")

Validation skipped. Set RUN_VALIDATION=1 to run a grouped holdout check.


# 10. Final LightGBM Training
### 10.1 Train the fast grouped ensemble

Train grouped LightGBM fold models, then add one full-data model to stabilize public-test predictions. This avoids model-directory reuse issues from heavier tabular frameworks and is lighter for repeated Kaggle runs.

In [10]:
X_train_model = train_model.drop(columns=["label"]).astype(np.float32)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(train_model["label"].astype(str))
sleep_classes = label_encoder.classes_
n_classes = len(sleep_classes)
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train_encoded)

unique_groups = groups.astype(str).nunique()
n_splits = max(2, min(LIGHTGBM_FOLDS, unique_groups, len(train_model)))
if unique_groups >= 2 and n_splits >= 2:
    splitter = GroupKFold(n_splits=n_splits)
    split_iter = splitter.split(X_train_model, y_train_encoded, groups=groups.astype(str))
    split_name = f"GroupKFold(n_splits={n_splits})"
else:
    n_splits = max(2, min(LIGHTGBM_FOLDS, int(pd.Series(y_train_encoded).value_counts().min())))
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    split_iter = splitter.split(X_train_model, y_train_encoded)
    split_name = f"StratifiedKFold(n_splits={n_splits})"

print("Training backend:", "LightGBM" if lgb is not None else "sklearn HistGradientBoosting fallback")
print("Validation splitter:", split_name)
print("Rows:", len(X_train_model), "Features:", X_train_model.shape[1], "Classes:", list(sleep_classes))

sleep_models = []
fold_rows = []
oof_proba = np.zeros((len(X_train_model), n_classes), dtype=np.float32)

for fold, (train_idx, valid_idx) in enumerate(split_iter, start=1):
    model = make_fast_sleep_model(SEED + fold, n_classes)
    fit_model(model, X_train_model.iloc[train_idx], y_train_encoded[train_idx], sample_weight=sample_weight[train_idx])
    fold_proba = aligned_predict_proba(model, X_train_model.iloc[valid_idx], n_classes)
    oof_proba[valid_idx] = fold_proba
    fold_pred = fold_proba.argmax(axis=1)
    fold_f1 = f1_score(y_train_encoded[valid_idx], fold_pred, average="weighted")
    fold_acc = accuracy_score(y_train_encoded[valid_idx], fold_pred)
    sleep_models.append(model)
    fold_rows.append({"fold": fold, "valid_rows": len(valid_idx), "weighted_f1": fold_f1, "accuracy": fold_acc})
    print(f"Fold {fold}: weighted_f1={fold_f1:.5f}, accuracy={fold_acc:.5f}, valid_rows={len(valid_idx)}")

# Add one full-data model to the ensemble. It is fast and usually stabilizes public-test predictions.
full_model = make_fast_sleep_model(SEED + 999, n_classes)
fit_model(full_model, X_train_model, y_train_encoded, sample_weight=sample_weight)
sleep_models.append(full_model)
sleep_model_weights = np.ones(len(sleep_models), dtype=np.float32)
sleep_model_weights[-1] = 1.0

valid_mask = oof_proba.sum(axis=1) > 0
if valid_mask.any():
    oof_pred = oof_proba[valid_mask].argmax(axis=1)
    oof_f1 = f1_score(y_train_encoded[valid_mask], oof_pred, average="weighted")
    oof_acc = accuracy_score(y_train_encoded[valid_mask], oof_pred)
    print(f"OOF weighted F1: {oof_f1:.5f}")
    print(f"OOF accuracy: {oof_acc:.5f}")

leaderboard = pd.DataFrame(fold_rows)
display(leaderboard)
print("Ensemble models:", len(sleep_models))

Training backend: LightGBM
Validation splitter: GroupKFold(n_splits=5)
Rows: 66745 Features: 999 Classes: ['N1', 'N2', 'N3', 'R', 'W']
Fold 1: weighted_f1=0.54152, accuracy=0.59805, valid_rows=13636
Fold 2: weighted_f1=0.61929, accuracy=0.64066, valid_rows=13636
Fold 3: weighted_f1=0.53031, accuracy=0.56641, valid_rows=12920
Fold 4: weighted_f1=0.56380, accuracy=0.58705, valid_rows=13636
Fold 5: weighted_f1=0.59749, accuracy=0.62569, valid_rows=12917
OOF weighted F1: 0.57113
OOF accuracy: 0.60373


,fold,valid_rows,weighted_f1,accuracy
0,1,13636,0.541525,0.598049
1,2,13636,0.619288,0.640657
2,3,12920,0.530311,0.566409
3,4,13636,0.563799,0.587049
4,5,12917,0.597493,0.625687


Ensemble models: 6


# 11. Build Test Features
### 11.1 Extract test features in sample submission order

Read each test segment by id, apply the same feature engineering pipeline, and align the resulting columns to the training matrix.

In [11]:
def infer_submission_columns(sample_df: pd.DataFrame):
    id_col = "id" if "id" in sample_df.columns else sample_df.columns[0]
    label_candidates = ["labels", "label", "Sleep_Stage", "target"]
    label_col = next((col for col in label_candidates if col in sample_df.columns), sample_df.columns[-1])
    if label_col == id_col and len(sample_df.columns) > 1:
        label_col = sample_df.columns[1]
    return id_col, label_col


def build_test_features(sample_df: pd.DataFrame, use_cache=True):
    id_col, _ = infer_submission_columns(sample_df)
    cache_path = FEATURE_CACHE_DIR / "test_features.pkl"
    sample_ids = sample_df[id_col].astype(str).tolist()

    if use_cache and cache_path.exists():
        cached = pd.read_pickle(cache_path)
        if "id" in cached.columns and cached["id"].astype(str).tolist() == sample_ids:
            print("Load cached test features:", cache_path)
            return cached

    rows = []
    missing = []
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Extract test features"):
        id_value = row[id_col]
        subject_id, segment_idx = split_test_id(id_value)
        try:
            segment_path = resolve_test_segment_path(id_value, DATA_DIR)
            segment_df = standardize_columns(pd.read_csv(segment_path))
            n_segments = np.nan
            if not pd.isna(segment_idx):
                n_segments = int(sample_df[id_col].astype(str).str.startswith(str(subject_id) + "_").sum())
            if pd.isna(segment_idx):
                segment_idx = 0
            if pd.isna(n_segments) or n_segments <= 0:
                n_segments = 1
            if len(segment_df) > WINDOW_SIZE:
                segment_df = segment_df.iloc[:WINDOW_SIZE].copy()
            features = extract_segment_features(segment_df, subject_id, int(segment_idx), int(n_segments))
            features["id"] = id_value
            features["segment_path"] = str(segment_path)
            rows.append(features)
        except Exception as exc:
            missing.append((id_value, repr(exc)))

    if missing:
        display(pd.DataFrame(missing, columns=["id", "error"]).head(30))
        raise RuntimeError(f"Failed to extract {len(missing)} test segments")

    test_features = pd.DataFrame(rows)
    test_features = add_sequence_context(test_features)
    test_features = test_features.replace([np.inf, -np.inf], np.nan)
    test_features.to_pickle(cache_path)
    print("Test feature shape:", test_features.shape)
    return test_features

test_features = build_test_features(sample_submission, use_cache=USE_FEATURE_CACHE)
test_model = apply_feature_imputer(test_features, feature_columns, feature_medians)[feature_columns]

display(test_features.head())
print("Test model shape:", test_model.shape)

Extract test features:   0%|          | 0/7832 [00:00<?, ?it/s]

Test feature shape: (7832, 1002)


,subject_id,segment_idx,n_segments,segment_pos,bvp_missing_frac,bvp_mean,bvp_std,bvp_var,bvp_min,bvp_max,bvp_median,bvp_q10,bvp_q25,bvp_q75,bvp_q90,bvp_iqr,bvp_range,bvp_mad,bvp_rms,bvp_energy,bvp_first,bvp_last,bvp_delta,bvp_slope,bvp_absdiff_mean,bvp_absdiff_std,bvp_absdiff_max,bvp_zcr,bvp_skew,bvp_kurtosis,bvp_fft_very_low_power,bvp_fft_very_low_rel,bvp_fft_low_power,bvp_fft_low_rel,bvp_fft_mid_power,bvp_fft_mid_rel,bvp_fft_high_power,bvp_fft_high_rel,bvp_fft_very_high_power,bvp_fft_very_high_rel,bvp_fft_dom_freq,bvp_fft_centroid,bvp_fft_entropy,ibi_missing_frac,ibi_mean,ibi_std,ibi_var,ibi_min,ibi_max,ibi_median,ibi_q10,ibi_q25,ibi_q75,ibi_q90,ibi_iqr,ibi_range,ibi_mad,ibi_rms,ibi_energy,ibi_first,...,acc_x_std_roll5_std,acc_x_std_roll5_delta,acc_x_std_roll10_mean,acc_x_std_roll10_std,acc_x_std_roll10_delta,acc_x_std_roll60_mean,acc_x_std_roll60_std,acc_x_std_roll60_delta,acc_y_std_subject_z,acc_y_std_subject_delta_mean,acc_y_std_lag1,acc_y_std_lead1,acc_y_std_diff_lag1,acc_y_std_diff_lead1,acc_y_std_lag2,acc_y_std_lead2,acc_y_std_diff_lag2,acc_y_std_diff_lead2,acc_y_std_lag3,acc_y_std_lead3,acc_y_std_diff_lag3,acc_y_std_diff_lead3,acc_y_std_roll3_mean,acc_y_std_roll3_std,acc_y_std_roll3_delta,acc_y_std_roll5_mean,acc_y_std_roll5_std,acc_y_std_roll5_delta,acc_y_std_roll10_mean,acc_y_std_roll10_std,acc_y_std_roll10_delta,acc_y_std_roll60_mean,acc_y_std_roll60_std,acc_y_std_roll60_delta,acc_z_std_subject_z,acc_z_std_subject_delta_mean,acc_z_std_lag1,acc_z_std_lead1,acc_z_std_diff_lag1,acc_z_std_diff_lead1,acc_z_std_lag2,acc_z_std_lead2,acc_z_std_diff_lag2,acc_z_std_diff_lead2,acc_z_std_lag3,acc_z_std_lead3,acc_z_std_diff_lag3,acc_z_std_diff_lead3,acc_z_std_roll3_mean,acc_z_std_roll3_std,acc_z_std_roll3_delta,acc_z_std_roll5_mean,acc_z_std_roll5_std,acc_z_std_roll5_delta,acc_z_std_roll10_mean,acc_z_std_roll10_std,acc_z_std_roll10_delta,acc_z_std_roll60_mean,acc_z_std_roll60_std,acc_z_std_roll60_delta
0,test001,0,814,0.00000,0.0,2.436632,96.985413,9406.170898,-502.470276,412.153778,8.922093,-104.330951,-37.086007,48.175610,77.324980,85.261617,914.624023,64.754318,97.016014,9412.107422,-16.787333,-54.305370,-37.518036,-0.023467,31.294769,36.701908,217.675781,0.133612,-0.229420,4.943695,1.919410e+06,0.002939,1.481707e+07,0.022691,263419104.0,0.403395,367871360.0,0.563351,4.978453e+06,0.007624,0.866667,1.158636,3.677973,0.0,0.962367,0.084580,0.007154,0.905438,1.151682,0.911322,0.911322,0.911322,1.019166,1.127755,0.107844,0.246244,0.072310,0.966077,0.933304,0.911322,...,0.845073,0.972618,0.685942,0.654515,1.102922,1.387965,3.085257,0.400900,0.195842,0.286150,NaN,0.285692,NaN,-0.392282,NaN,0.190395,NaN,-0.487579,NaN,0.151619,NaN,-0.526354,0.481833,0.277385,0.196141,0.384687,0.258425,0.293287,0.309015,0.212347,0.368959,0.740481,1.820035,-0.062507,0.030731,0.098979,NaN,0.168267,NaN,-0.520612,NaN,0.190744,NaN,-0.498135,NaN,0.234401,NaN,-0.454479,0.428573,0.368128,0.260306,0.349297,0.294302,0.339582,0.343412,0.219978,0.345467,0.658108,0.781988,0.030771
1,test001,1,814,0.00123,0.0,-0.224025,52.860653,2794.248779,-158.916641,71.545494,15.325783,-82.444057,-33.858389,42.803202,56.009967,76.661591,230.462128,43.977619,52.861130,2794.299072,-87.709602,-92.061218,-4.351616,0.003124,19.410105,17.315041,78.974716,0.116910,-0.833462,-0.317551,4.886053e+05,0.004069,4.939055e+05,0.004114,93951816.0,0.782501,24337292.0,0.202699,7.944315e+05,0.006617,0.900000,1.141972,2.088076,0.0,1.094589,0.041303,0.001706,1.009564,1.147701,1.111726,1.034764,1.050544,1.127643,1.142997,0.077099,0.138137,0.037437,1.095367,1.199830,1.128544,...,0.753749,-0.266364,0.771404,0.621719,-0.373212,1.361037,3.037103,-0.962844,-0.072637,-0.106132,0.677974,0.190395,-0.392282,-0.095297,NaN,0.151619,NaN,-0.134073,NaN,0.239396,NaN,-0.046296,0.384687,0.258425,-0.098995,0.326420,0.241044,-0.040728,0.352096,0.217276,-0.066404,0.727637,1.790872,-0.441945,-0.130907,-0.421633,0.688879,0.190744,-0.520612,0.022477,NaN,0.234401,NaN,0.066134,NaN,0.434770,NaN,0.266503,0.349297,0.294302,-0.1

Test model shape: (7832, 999)


# 12. Prediction & Submission Generation
### 12.1 Predict labels and write submission

Average probabilities from the LightGBM ensemble, optionally smooth each subject timeline, and save the final file in the exact sample submission format.

In [12]:
def smooth_prediction_probabilities(proba_df: pd.DataFrame, ids: pd.Series, window: int = 3) -> pd.DataFrame:
    proba = proba_df.copy().reset_index(drop=True)
    class_columns = list(proba.columns)

    helper = pd.DataFrame({"id": ids.astype(str).to_numpy(), "_original_order": np.arange(len(proba))})
    parsed = helper["id"].apply(split_test_id)
    helper["_subject_id"] = parsed.apply(lambda x: str(x[0]))
    helper["_segment_idx"] = parsed.apply(lambda x: -1 if pd.isna(x[1]) else int(x[1]))
    helper = pd.concat([helper, proba], axis=1)
    helper = helper.sort_values(["_subject_id", "_segment_idx", "_original_order"]).reset_index(drop=True)

    for _, idx in helper.groupby("_subject_id", sort=False).groups.items():
        smoothed = helper.loc[idx, class_columns].rolling(window=window, min_periods=1, center=True).mean()
        helper.loc[idx, class_columns] = smoothed.to_numpy()

    return helper.sort_values("_original_order")[class_columns].reset_index(drop=True)


def labels_from_probabilities(proba_df: pd.DataFrame) -> np.ndarray:
    return proba_df.idxmax(axis=1).to_numpy()


def ensemble_predict_proba(models, X, n_classes: int, weights=None) -> np.ndarray:
    if weights is None:
        weights = np.ones(len(models), dtype=np.float32)
    weights = np.asarray(weights, dtype=np.float32)
    weights = weights / max(float(weights.sum()), EPS)

    proba = np.zeros((len(X), n_classes), dtype=np.float32)
    X = X.astype(np.float32)
    for weight, model in zip(weights, models):
        proba += weight * aligned_predict_proba(model, X, n_classes)
    return proba


def smooth_isolated_predictions(submission_df: pd.DataFrame, id_col: str, label_col: str) -> pd.DataFrame:
    out = submission_df.copy()
    helper = out[[id_col, label_col]].copy()
    parsed = helper[id_col].apply(split_test_id)
    helper["_subject_id"] = parsed.apply(lambda x: str(x[0]))
    helper["_segment_idx"] = parsed.apply(lambda x: -1 if pd.isna(x[1]) else int(x[1]))
    helper["_original_order"] = np.arange(len(helper))
    helper = helper.sort_values(["_subject_id", "_segment_idx", "_original_order"]).reset_index(drop=True)

    for _, idx in helper.groupby("_subject_id", sort=False).groups.items():
        labels = helper.loc[idx, label_col].tolist()
        smoothed = labels.copy()
        for pos in range(1, len(labels) - 1):
            if labels[pos - 1] == labels[pos + 1] and labels[pos] != labels[pos - 1]:
                smoothed[pos] = labels[pos - 1]
        helper.loc[idx, label_col] = smoothed

    helper = helper.sort_values("_original_order")
    out[label_col] = helper[label_col].to_numpy()
    return out


id_col, label_col = infer_submission_columns(sample_submission)
raw_proba_array = ensemble_predict_proba(sleep_models, test_model, n_classes, sleep_model_weights)
raw_proba = pd.DataFrame(raw_proba_array, columns=sleep_classes)

raw_submission = sample_submission.copy()
raw_submission[label_col] = labels_from_probabilities(raw_proba)
raw_submission.to_csv(RAW_SUBMISSION_PATH, index=False)

final_proba = raw_proba
if SMOOTH_PROBABILITIES:
    final_proba = smooth_prediction_probabilities(raw_proba, sample_submission[id_col], window=3)

submission = sample_submission.copy()
submission[label_col] = labels_from_probabilities(final_proba)

if SMOOTH_ISOLATED_STAGES:
    submission = smooth_isolated_predictions(submission, id_col, label_col)

assert len(submission) == len(sample_submission)
assert submission[label_col].notna().all()
submission.to_csv(SUBMISSION_PATH, index=False)

print("Raw submission:", RAW_SUBMISSION_PATH)
print("Final submission:", SUBMISSION_PATH)
print("Probability smoothing:", SMOOTH_PROBABILITIES)
print("Isolated label smoothing:", SMOOTH_ISOLATED_STAGES)
display(submission[label_col].value_counts(dropna=False))
display(submission.head())

Raw submission: /kaggle/working/sleep_stage_lv2_raw_submission.csv
Final submission: /kaggle/working/sleep_stage_lv2_submission.csv
Probability smoothing: True
Isolated label smoothing: True


labels
N2    5584
W     1759
N1     325
R      142
N3      22
Name: count, dtype: int64

,id,labels
0,test001_00000,W
1,test001_00001,W
2,test001_00002,W
3,test001_00003,W
4,test001_00004,W
